In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import logging
import sys
import warnings
from typing import Dict, Optional, Tuple

from scipy.stats import spearmanr, pearsonr
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score, mean_absolute_error

# Add project root to sys.path (robust)
from pathlib import Path as _Path

HERE = _Path.cwd().resolve()
root = None
for p in [HERE] + list(HERE.parents):
    if (p / "cgt_xai").is_dir():
        root = p
        break
if root is None:
    raise RuntimeError(f"Could not find project root containing 'cgt_xai' from {HERE}")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Project root added to sys.path:", root)

# Project imports
from cgt_xai.models import get_model
from cgt_xai.utils.plotting import plot_single_valuation

# Attribution methods
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.least_core import montecarlo_least_core
from cgt_xai.concepts.shapley import montecarlo_shapley_value

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Config
DATASETS = ["mi_complications"]  # add "lung_cancer" if desired
MODEL_NAMES = ["svm", "logistic", "random_forest"]
SCORING = "accuracy"
RANDOM_STATE = 42

BASE_OUTPUT_DIR = Path("./output/feature_valuation_uci_paper_eval_multi_master_corrected")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Curve settings (masking-based => can be higher without huge cost)
CURVE_STEPS = 50

# Robustness settings
NOISE_LEVEL = 0.05
BOOTSTRAP_ROUNDS = 10

# Shapley settings
SHAPLEY_MC_ITERS = 100

# Least-core settings
LEAST_CORE_MC_ITERS = 300
LEAST_CORE_EPSILON = 1e-6

LEAST_CORE_SOLVER_OPTS = {
    "solver": "SCS",
    "max_iters": 25000,
    "eps": 1e-6,
    "verbose": False
}

# Faithfulness-corr settings
FC_N_TESTS = 100
FC_SUBSET_SIZE = 20

# Dataset loaders
def load_lung_cancer():
    path = "../../datasets/lung-cancer.data"
    df = pd.read_csv(path, header=None, na_values="?")

    y = df.iloc[:, 0].astype(int).values
    X_df = df.iloc[:, 1:]
    feature_names = [f"attr_{i}" for i in range(1, X_df.shape[1] + 1)]

    imputer = SimpleImputer(strategy="most_frequent")
    X = imputer.fit_transform(X_df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
    )
    return X_train, X_test, y_train, y_test, feature_names


def load_mi_complications(binary_let_is=True):
    path = "../../datasets/MI.data"
    df = pd.read_csv(path, header=None, na_values="?")

    X_df = df.iloc[:, 1:112]
    feature_names = [f"feat_{i}" for i in range(1, X_df.shape[1] + 1)]

    let_is = df.iloc[:, 123].astype(int).values
    y = (let_is != 0).astype(int) if binary_let_is else let_is

    imputer = SimpleImputer(strategy="most_frequent")
    X = imputer.fit_transform(X_df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    return X_train, X_test, y_train, y_test, feature_names


def load_dataset(name: str):
    if name == "lung_cancer":
        return load_lung_cancer()
    elif name == "mi_complications":
        return load_mi_complications(binary_let_is=True)
    else:
        raise ValueError(f"Unknown dataset name: {name}")

# Utility + scorer (retraining game)
class DataWrapper:
    def __init__(self, indices, names=None):
        self.indices = indices
        self.data_names = names if names is not None else [f"f{i}" for i in range(len(indices))]

    def __len__(self):
        return len(self.indices)


class Utility:
    """
    v(S): train model on features S, evaluate on test.
    Includes caching for speed.
    """
    def __init__(self, X_train, y_train, X_test, y_test, model_name, scorer,
                 feature_names=None, random_state=42):
        self.X_train = np.asarray(X_train)
        self.y_train = np.asarray(y_train)
        self.X_test = np.asarray(X_test)
        self.y_test = np.asarray(y_test)
        self.model_name = model_name
        self.scorer = scorer
        self.data = DataWrapper(list(range(self.X_train.shape[1])), feature_names)
        self.random_state = random_state
        self._cache = {}

    def __call__(self, subset):
        key = tuple(sorted(subset))
        if key in self._cache:
            return self._cache[key]

        if len(key) == 0:
            majority = np.bincount(self.y_train.astype(int)).argmax()
            y_pred = np.full_like(self.y_test, majority)
            score = self.scorer(self.y_test, y_pred)
            self._cache[key] = float(score)
            return float(score)

        model = get_model(self.model_name, self.random_state)
        model.fit(self.X_train[:, key], self.y_train)
        y_pred = model.predict(self.X_test[:, key])
        score = self.scorer(self.y_test, y_pred)
        self._cache[key] = float(score)
        return float(score)


def get_scorer(name):
    if name == "accuracy":
        return lambda y_true, y_pred: accuracy_score(y_true, y_pred)
    elif name == "f1_macro":
        return lambda y_true, y_pred: f1_score(y_true, y_pred, average="macro")
    elif name == "r2":
        return lambda y_true, y_pred: r2_score(y_true, y_pred)
    elif name == "mse":
        return lambda y_true, y_pred: -mean_squared_error(y_true, y_pred)
    elif name == "rmse":
        return lambda y_true, y_pred: -np.sqrt(mean_squared_error(y_true, y_pred))
    elif name == "mae":
        return lambda y_true, y_pred: -mean_absolute_error(y_true, y_pred)
    else:
        raise ValueError(f"Unknown scorer: {name}")

# LOO impacts (retraining paradigm)
def loo_impacts(utility: Utility):
    n = len(utility.data)
    all_set = set(range(n))
    vN = utility(list(all_set))
    impacts = np.zeros(n, dtype=float)
    for i in range(n):
        vN_minus_i = utility(list(all_set - {i}))
        impacts[i] = vN - vN_minus_i
    return float(vN), impacts

# Robustness helpers (same as before)
def cosine(a, b):
    return float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-12))


def noise_stability_cosine(base_utility: Utility, method_fn, noise_level: float = 0.05):
    phi0 = method_fn(base_utility)

    X_train = base_utility.X_train
    noise = np.random.normal(0, noise_level * X_train.std(axis=0), X_train.shape)
    X_noisy = X_train + noise

    util_noisy = Utility(
        X_noisy, base_utility.y_train, base_utility.X_test, base_utility.y_test,
        model_name=base_utility.model_name,
        scorer=base_utility.scorer,
        feature_names=base_utility.data.data_names,
        random_state=base_utility.random_state + 11
    )
    phi_noisy = method_fn(util_noisy)
    return cosine(phi0, phi_noisy)


def bootstrap_stability_cosine(base_utility: Utility, method_fn, rounds: int = 10):
    X_train = base_utility.X_train
    y_train = base_utility.y_train
    X_test = base_utility.X_test
    y_test = base_utility.y_test

    phi0 = method_fn(base_utility)

    rng = np.random.default_rng(base_utility.random_state)
    n = X_train.shape[0]
    cos_list = []
    for r in range(rounds):
        idx = rng.integers(0, n, size=n)
        Xb = X_train[idx]
        yb = y_train[idx]

        util_b = Utility(
            Xb, yb, X_test, y_test,
            model_name=base_utility.model_name,
            scorer=base_utility.scorer,
            feature_names=base_utility.data.data_names,
            random_state=base_utility.random_state + 1000 + r
        )
        phi_b = method_fn(util_b)
        cos_list.append(cosine(phi0, phi_b))

    return float(np.mean(cos_list))

# Attribution methods (same)
def gately_method_fn(utility: Utility) -> np.ndarray:
    res = exact_gately_value(utility, options=None, progress=False)
    return np.array(res.values, dtype=float)


def least_core_mc_method_fn(utility: Utility) -> np.ndarray:
    res = montecarlo_least_core(
        utility,
        n_iterations=LEAST_CORE_MC_ITERS,
        epsilon=LEAST_CORE_EPSILON,
        options=LEAST_CORE_SOLVER_OPTS,
        progress=False
    )
    return np.array(res.values, dtype=float)


def shapley_mc_method_fn(utility: Utility) -> np.ndarray:
    res = montecarlo_shapley_value(
        utility,
        n_iterations=SHAPLEY_MC_ITERS,
        options=None,
        progress=False
    )
    return np.array(res.values, dtype=float)


ATTRIBUTION_METHODS = {
    "gately": gately_method_fn,
    "least_core_mc": least_core_mc_method_fn,
    "mc_shapley": shapley_mc_method_fn,
}

# FAST masking-based multi-metric faithfulness evaluator
class RobustFaithfulnessEvaluator:
    """
    Computes masking-based faithfulness metrics efficiently:
      - deletion_auc_drop (vs random)
      - insertion_auc_gain (vs random)
      - uncertainty_discriminative (if predict_proba)
      - faithfulness_corr (subset impact vs avg attribution)
      - sufficiency_{80,90,95}
    Plus LOO correlations provided separately.
    """

    def __init__(self, model, X_train, X_test, y_test, scorer):
        if model is None:
            raise ValueError("model must be a trained estimator.")
        self.model = model
        self.X_train = np.asarray(X_train)
        self.X_test = np.asarray(X_test)
        self.y_test = np.asarray(y_test)
        self.scorer = scorer

        # robust baseline
        self.baseline_vec = np.median(self.X_train, axis=0)

        self._has_proba = hasattr(model, "predict_proba") and callable(getattr(model, "predict_proba"))
        self._has_decision = hasattr(model, "decision_function") and callable(getattr(model, "decision_function"))

        self.y_pred_orig = None
        self.y_proba_orig = None
        self.y_decision_orig = None
        self._cache_orig_outputs()

    def _cache_orig_outputs(self):
        try:
            self.y_pred_orig = self.model.predict(self.X_test)
        except Exception as e:
            warnings.warn(f"predict failed on X_test: {e}")

        if self._has_proba:
            try:
                self.y_proba_orig = self.model.predict_proba(self.X_test)
            except Exception as e:
                warnings.warn(f"predict_proba failed; disabling proba metrics. Error: {e}")
                self._has_proba = False
                self.y_proba_orig = None

        if (not self._has_proba) and self._has_decision:
            try:
                self.y_decision_orig = self.model.decision_function(self.X_test)
            except Exception as e:
                warnings.warn(f"decision_function failed; disabling decision metrics. Error: {e}")
                self._has_decision = False
                self.y_decision_orig = None

    def _masking_curve_auc(self, order: np.ndarray, steps: int, mode: str) -> float:
        fracs = np.linspace(0, 1, steps)
        scores = []
        n_feat = len(order)
        n_samp = self.X_test.shape[0]

        for f in fracs:
            k = int(round(f * n_feat))
            if k <= 0:
                if mode == "deletion":
                    X_masked = self.X_test.copy()
                else:
                    X_masked = np.tile(self.baseline_vec, (n_samp, 1))
            else:
                cols = order[:k]
                if mode == "deletion":
                    X_masked = self.X_test.copy()
                    X_masked[:, cols] = self.baseline_vec[cols]
                else:
                    X_masked = np.tile(self.baseline_vec, (n_samp, 1))
                    X_masked[:, cols] = self.X_test[:, cols]

            y_pred = self.model.predict(X_masked)
            scores.append(float(self.scorer(self.y_test, y_pred)))

        return float(np.trapezoid(np.array(scores), fracs))

    def deletion_auc_drop(self, attribution: np.ndarray, steps: int) -> float:
        order_attr = np.argsort(-np.abs(attribution))
        order_rand = np.random.default_rng(42).permutation(len(attribution))
        auc_attr = self._masking_curve_auc(order_attr, steps, "deletion")
        auc_rand = self._masking_curve_auc(order_rand, steps, "deletion")
        return float(auc_rand - auc_attr)

    def insertion_auc_gain(self, attribution: np.ndarray, steps: int) -> float:
        order_attr = np.argsort(-np.abs(attribution))
        order_rand = np.random.default_rng(42).permutation(len(attribution))
        auc_attr = self._masking_curve_auc(order_attr, steps, "insertion")
        auc_rand = self._masking_curve_auc(order_rand, steps, "insertion")
        return float(auc_attr - auc_rand)

    def uncertainty_discriminative(self, attribution: np.ndarray, steps: int = 10) -> float:
        if not self._has_proba:
            return float("nan")
        order_attr = np.argsort(-np.abs(attribution))
        order_rand = np.random.default_rng(42).permutation(len(attribution))
        auc_attr = self._uncertainty_auc(order_attr, steps)
        auc_rand = self._uncertainty_auc(order_rand, steps)
        return float(auc_attr - auc_rand)

    def _uncertainty_auc(self, order: np.ndarray, steps: int) -> float:
        fracs = np.linspace(0.1, 1.0, steps)
        vals = []
        for f in fracs:
            k = max(1, int(round(f * len(order))))
            cols = order[:k]
            X_masked = self.X_test.copy()
            X_masked[:, cols] = self.baseline_vec[cols]
            y_proba = self.model.predict_proba(X_masked)
            vals.append(float((1.0 - y_proba.max(axis=1)).mean()))
        return float(np.trapezoid(np.array(vals), fracs))

    def faithfulness_corr(self, attribution: np.ndarray, n_tests: int = 100, subset_size: int = 20) -> float:
        abs_attr = np.abs(attribution)
        n_feat = len(attribution)
        rng = np.random.default_rng(42)
        k = min(int(subset_size), max(1, n_feat // 2))

        avg_imp = np.empty(n_tests, dtype=float)
        drops = np.empty(n_tests, dtype=float)
        eps = 1e-10
        idx = np.arange(self.y_test.shape[0])

        for t in range(n_tests):
            subset = rng.choice(n_feat, size=k, replace=False)
            avg_imp[t] = float(abs_attr[subset].mean())

            X_masked = self.X_test.copy()
            X_masked[:, subset] = self.baseline_vec[subset]

            if self._has_proba:
                ypm = self.model.predict_proba(X_masked)
                p0 = self.y_proba_orig[idx, self.y_test]
                p1 = ypm[idx, self.y_test]
                if ypm.shape[1] == 2:
                    # correct log-odds
                    logit0 = np.log(p0 + eps) - np.log(1.0 - p0 + eps)
                    logit1 = np.log(p1 + eps) - np.log(1.0 - p1 + eps)
                else:
                    # multiclass log-prob
                    logit0 = np.log(p0 + eps)
                    logit1 = np.log(p1 + eps)
                drops[t] = float((logit0 - logit1).mean())

            elif self._has_decision:
                dm = self.model.decision_function(X_masked)
                d0 = self.y_decision_orig
                if dm.ndim == 1:
                    # binary signed margin
                    sign = np.where(self.y_test == 1, 1.0, -1.0)
                    drops[t] = float((sign * d0 - sign * dm).mean())
                else:
                    drops[t] = float((d0[idx, self.y_test] - dm[idx, self.y_test]).mean())
            else:
                return float("nan")

        corr, _ = pearsonr(avg_imp, drops)
        return float(corr)

    def sufficiency_multi(self, attribution: np.ndarray, thresholds=(0.8, 0.9, 0.95)) -> Dict[str, float]:
        abs_attr = np.abs(attribution)
        order = np.argsort(-abs_attr)
        n_samp = self.X_test.shape[0]

        full_score = float(self.scorer(self.y_test, self.model.predict(self.X_test)))
        out: Dict[str, float] = {}

        for thr in thresholds:
            target = thr * full_score
            found = False
            for k in range(1, len(attribution) + 1):
                cols = order[:k]
                X_masked = np.tile(self.baseline_vec, (n_samp, 1))
                X_masked[:, cols] = self.X_test[:, cols]
                score = float(self.scorer(self.y_test, self.model.predict(X_masked)))
                if score >= target:
                    out[f"sufficiency_{int(thr*100)}"] = float(k / len(attribution))
                    found = True
                    break
            if not found:
                out[f"sufficiency_{int(thr*100)}"] = 1.0
        return out

    def evaluate_masking_metrics(self, attribution: np.ndarray, steps: int) -> Dict[str, float]:
        res: Dict[str, float] = {}
        res["deletion_auc_drop"] = self.deletion_auc_drop(attribution, steps)
        res["insertion_auc_gain"] = self.insertion_auc_gain(attribution, steps)
        if self._has_proba:
            res["uncertainty_discriminative"] = self.uncertainty_discriminative(attribution, steps=max(8, steps // 5))
        else:
            res["uncertainty_discriminative"] = np.nan
        res["faithfulness_corr"] = self.faithfulness_corr(attribution, n_tests=FC_N_TESTS, subset_size=FC_SUBSET_SIZE)
        res.update(self.sufficiency_multi(attribution, thresholds=(0.8, 0.9, 0.95)))
        return res

# Ranking + summaries (exclude p-values/metadata)
def rank_methods(metric_df: pd.DataFrame, higher_is_better_cols):
    rank_df = pd.DataFrame(index=metric_df.index)
    for col in metric_df.columns:
        higher = col in higher_is_better_cols
        rank_df[col] = metric_df[col].rank(ascending=not higher, method="average")
    return rank_df


def rank_summary(rank_df: pd.DataFrame):
    return pd.DataFrame({
        "mean_rank_mu_hat": rank_df.mean(axis=1),
        "std_rank_sigma_hat": rank_df.std(axis=1, ddof=0)
    }).sort_values("mean_rank_mu_hat")

# Main
def main():
    scorer = get_scorer(SCORING)

    master_rows_faith = []
    master_rows_rob = []
    master_rows_faith_rank = []
    master_rows_rob_rank = []
    master_rows_faith_summary = []
    master_rows_rob_summary = []

    for dataset_name in DATASETS:
        X_train, X_test, y_train, y_test, feature_names = load_dataset(dataset_name)

        for model_name in MODEL_NAMES:
            logger.info("\n" + "=" * 70)
            logger.info(f"DATASET: {dataset_name} | MODEL: {model_name}")
            logger.info("=" * 70)

            OUTPUT_DIR = BASE_OUTPUT_DIR / f"{dataset_name}_{model_name}_results"
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

            # retraining-game utility
            utility = Utility(
                X_train, y_train, X_test, y_test,
                model_name=model_name,
                scorer=scorer,
                feature_names=feature_names,
                random_state=RANDOM_STATE
            )

            # Train full model ONCE for masking metrics
            full_model = get_model(model_name, RANDOM_STATE)
            full_model.fit(X_train, y_train)

            evaluator = RobustFaithfulnessEvaluator(
                full_model, X_train, X_test, y_test, scorer
            )

            vN, impacts = loo_impacts(utility)
            logger.info(f"  v(N) with all features ({SCORING}) = {vN:.4f}")

            # Compute attributions (retraining game)
            attributions = {}
            for mname, mfn in ATTRIBUTION_METHODS.items():
                logger.info(f"Computing attributions: {mname}")
                phi = mfn(utility)
                attributions[mname] = phi

                df_vals = pd.DataFrame({"method": mname, "feature": feature_names, "value": phi})
                df_vals.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_feature_valuations_{mname}.csv", index=False)

                try:
                    plot_single_valuation(
                        phi, feature_names,
                        method_name=mname,
                        title=f"{mname} - {dataset_name}_{model_name} ({SCORING})",
                        output_path=OUTPUT_DIR / f"{dataset_name}_{model_name}_{mname}.png"
                    )
                except Exception as e:
                    logger.warning(f"Plot failed for {mname}: {e}")

            # Faithfulness metrics (MULTI-METRIC, efficient)
            faith_rows = []
            for mname, phi in attributions.items():
                # LOO correlations (retraining paradigm)
                sp, p_sp = spearmanr(phi, impacts)
                pe, p_pe = pearsonr(phi, impacts)

                mask_metrics = evaluator.evaluate_masking_metrics(phi, steps=CURVE_STEPS)

                row = {
                    "dataset": dataset_name,
                    "model": model_name,
                    "method": mname,
                    "loo_spearman": float(sp),
                    "loo_pearson": float(pe),
                    "_p_spearman": float(p_sp),
                    "_p_pearson": float(p_pe),
                    "vN": float(vN),
                    **mask_metrics,
                }
                faith_rows.append(row)
                master_rows_faith.append(row)

            faith_df = pd.DataFrame(faith_rows).set_index("method")
            faith_df.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_faithfulness_metrics_raw.csv")
            logger.info("\nFaithfulness metrics (raw):\n" + faith_df.to_string())

            rob_rows = []
            for mname, mfn in ATTRIBUTION_METHODS.items():
                noise_cos = noise_stability_cosine(utility, mfn, noise_level=NOISE_LEVEL)
                boot_cos = bootstrap_stability_cosine(utility, mfn, rounds=BOOTSTRAP_ROUNDS)
                row = {
                    "dataset": dataset_name,
                    "model": model_name,
                    "method": mname,
                    "noise_cosine": float(noise_cos),
                    "bootstrap_cosine": float(boot_cos),
                }
                rob_rows.append(row)
                master_rows_rob.append(row)

            rob_df = pd.DataFrame(rob_rows).set_index("method")
            rob_df.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_robustness_metrics_raw.csv")
            logger.info("\nRobustness metrics (raw):\n" + rob_df.to_string())

            # Ranking
            # Exclude metadata columns starting with '_'
            faith_metric_cols = [c for c in faith_df.columns if not c.startswith("_") and c not in ["dataset", "model", "vN"]]
            rob_metric_cols = ["noise_cosine", "bootstrap_cosine"]

            faith_rank = rank_methods(faith_df[faith_metric_cols], higher_is_better_cols=set(
                [c for c in faith_metric_cols if not c.startswith("sufficiency_")]
            ))
            # sufficiency_* lower is better => remove from higher_is_better set
            # and then ranking those columns with ascending=True (lower better)
            # Easiest: recompute properly per-col:
            faith_rank = pd.DataFrame(index=faith_df.index)
            for col in faith_metric_cols:
                higher = not col.startswith("sufficiency_")
                faith_rank[col] = faith_df[col].rank(ascending=not higher, method="average")

            rob_rank = rank_methods(rob_df[rob_metric_cols], higher_is_better_cols=set(rob_metric_cols))

            faith_rank.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_faithfulness_ranks.csv")
            rob_rank.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_robustness_ranks.csv")

            for method in faith_rank.index:
                r = {"dataset": dataset_name, "model": model_name, "method": method}
                r.update({f"rank_{c}": float(faith_rank.loc[method, c]) for c in faith_metric_cols})
                master_rows_faith_rank.append(r)

            for method in rob_rank.index:
                r = {"dataset": dataset_name, "model": model_name, "method": method}
                r.update({f"rank_{c}": float(rob_rank.loc[method, c]) for c in rob_metric_cols})
                master_rows_rob_rank.append(r)

            # Mean rank μ̂ and disagreement σ̂
            faith_summary = rank_summary(faith_rank)
            rob_summary = rank_summary(rob_rank)

            faith_summary.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_faithfulness_rank_summary.csv")
            rob_summary.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_robustness_rank_summary.csv")

            logger.info("\nFaithfulness rank summary (μ̂, σ̂):\n" + faith_summary.to_string())
            logger.info("\nRobustness rank summary (μ̂, σ̂):\n" + rob_summary.to_string())

            for method in faith_summary.index:
                master_rows_faith_summary.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "method": method,
                    "faith_mean_rank_mu_hat": float(faith_summary.loc[method, "mean_rank_mu_hat"]),
                    "faith_std_rank_sigma_hat": float(faith_summary.loc[method, "std_rank_sigma_hat"]),
                })
            for method in rob_summary.index:
                master_rows_rob_summary.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "method": method,
                    "rob_mean_rank_mu_hat": float(rob_summary.loc[method, "mean_rank_mu_hat"]),
                    "rob_std_rank_sigma_hat": float(rob_summary.loc[method, "std_rank_sigma_hat"]),
                })

            logger.info(f"\nSaved per-run outputs to: {OUTPUT_DIR}")
            logger.info("=" * 70)

    # MASTER CSVs
    master_faith = pd.DataFrame(master_rows_faith)
    master_rob = pd.DataFrame(master_rows_rob)
    master_faith_rank = pd.DataFrame(master_rows_faith_rank)
    master_rob_rank = pd.DataFrame(master_rows_rob_rank)
    master_faith_summary = pd.DataFrame(master_rows_faith_summary)
    master_rob_summary = pd.DataFrame(master_rows_rob_summary)

    paper_master = master_faith.merge(master_rob, on=["dataset", "model", "method"], how="outer")
    paper_master = paper_master.merge(master_faith_summary, on=["dataset", "model", "method"], how="outer")
    paper_master = paper_master.merge(master_rob_summary, on=["dataset", "model", "method"], how="outer")

    master_faith_path = BASE_OUTPUT_DIR / "MASTER_faithfulness_raw.csv"
    master_rob_path = BASE_OUTPUT_DIR / "MASTER_robustness_raw.csv"
    master_faith_rank_path = BASE_OUTPUT_DIR / "MASTER_faithfulness_ranks.csv"
    master_rob_rank_path = BASE_OUTPUT_DIR / "MASTER_robustness_ranks.csv"
    paper_master_path = BASE_OUTPUT_DIR / "MASTER_paper_table.csv"

    master_faith.to_csv(master_faith_path, index=False)
    master_rob.to_csv(master_rob_path, index=False)
    master_faith_rank.to_csv(master_faith_rank_path, index=False)
    master_rob_rank.to_csv(master_rob_rank_path, index=False)
    paper_master.to_csv(paper_master_path, index=False)

    logger.info("\nMASTER CSVs saved:")
    logger.info(f"  {master_faith_path}")
    logger.info(f"  {master_rob_path}")
    logger.info(f"  {master_faith_rank_path}")
    logger.info(f"  {master_rob_rank_path}")
    logger.info(f"  {paper_master_path}")


if __name__ == "__main__":
    main()

✅ Project root added to sys.path: 


In [ ]:
"""
COMPLETE, CORRECT, WORKING SCRIPT (single drop-in file)

This version fixes ALL the issues you hit:
1) No more IndexError / tiling mismatch (works for ANY PERTURB_BATCH, even when not multiple of n_test)
2) Faithfulness-corr uses sum(|phi|) and FC-like signal (logit/logprob/margin) under SAME deletion protocol
3) Sufficiency uses positive curve score (p_ref / margin) so it won't get stuck at 1.0 due to negative logs
4) Masking baselines use TRAIN ROW sampling (more on-manifold than per-column sampling)
5) REF_MODE configurable: "pred" (model faithfulness) or "true" (ground-truth reference)
6) Least-core solver configurable by a single SOLVER_NAME variable (default "SCS")

Everything else is your original pipeline:
- retraining-game Utility v(S)
- LOO impacts
- attribution methods: gately, least_core_mc, mc_shapley
- robustness metrics
- rankings and master CSVs

Just run this file.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import logging
import sys
import warnings
from typing import Dict, Optional, Literal

from scipy.stats import spearmanr, pearsonr
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, mean_squared_error, r2_score, mean_absolute_error
)

# Add project root to sys.path (robust)
from pathlib import Path as _Path

HERE = _Path.cwd().resolve()
root = None
for p in [HERE] + list(HERE.parents):
    if (p / "cgt_xai").is_dir():
        root = p
        break
if root is None:
    raise RuntimeError(f"Could not find project root containing 'cgt_xai' from {HERE}")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Project root added to sys.path:", root)

# Project imports
from cgt_xai.models import get_model
from cgt_xai.utils.plotting import plot_single_valuation

from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.least_core import montecarlo_least_core
from cgt_xai.concepts.shapley import montecarlo_shapley_value

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Config
DATASETS = ["mi_complications"]  # add "lung_cancer" if desired
MODEL_NAMES = ["svm", "logistic", "random_forest"]
SCORING = "accuracy"  # ONLY for retraining-game utility/LOO
RANDOM_STATE = 42

BASE_OUTPUT_DIR = Path("./output/feature_valuation_uci_paper_eval_multi_master_corrected_FINAL")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Masking curve settings
CURVE_STEPS = 10

# Robustness settings
NOISE_LEVEL = 0.05
BOOTSTRAP_ROUNDS = 10

# Shapley MC settings
SHAPLEY_MC_ITERS = 100

# Least-core settings
LEAST_CORE_MC_ITERS = 300
LEAST_CORE_EPSILON = 1e-6

SOLVER_NAME = "SCS"
LEAST_CORE_SOLVER_OPTS = {
    "solver": SOLVER_NAME,
    "max_iters": 25000,
    "eps": 1e-6,
    "verbose": False
}

# Faithfulness-corr settings
FC_N_TESTS = 100
FC_SUBSET_SIZE = 20

# Masking perturbation sampling
PERTURB_N_BASELINES = 8       # MC averaging baselines per step
PERTURB_BATCH = 2048          # can be ANY number now
REF_MODE: Literal["pred", "true"] = "pred"  # "pred" recommended (model-faithfulness); "true" for ground truth ref


# Dataset loaders
def load_lung_cancer():
    path = "../../datasets/lung-cancer.data"
    df = pd.read_csv(path, header=None, na_values="?")

    y = df.iloc[:, 0].astype(int).values
    X_df = df.iloc[:, 1:]
    feature_names = [f"attr_{i}" for i in range(1, X_df.shape[1] + 1)]

    imputer = SimpleImputer(strategy="most_frequent")
    X = imputer.fit_transform(X_df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
    )
    return X_train, X_test, y_train, y_test, feature_names


def load_mi_complications(binary_let_is=True):
    path = "../../datasets/MI.data"
    df = pd.read_csv(path, header=None, na_values="?")

    X_df = df.iloc[:, 1:112]
    feature_names = [f"feat_{i}" for i in range(1, X_df.shape[1] + 1)]

    let_is = df.iloc[:, 123].astype(int).values
    y = (let_is != 0).astype(int) if binary_let_is else let_is

    imputer = SimpleImputer(strategy="most_frequent")
    X = imputer.fit_transform(X_df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    return X_train, X_test, y_train, y_test, feature_names


def load_dataset(name: str):
    if name == "lung_cancer":
        return load_lung_cancer()
    if name == "mi_complications":
        return load_mi_complications(binary_let_is=True)
    raise ValueError(f"Unknown dataset name: {name}")


# Utility + scorer (retraining game)
class DataWrapper:
    def __init__(self, indices, names=None):
        self.indices = indices
        self.data_names = names if names is not None else [f"f{i}" for i in range(len(indices))]

    def __len__(self):
        return len(self.indices)


class Utility:
    """
    v(S): train model on features S, evaluate on test.
    Includes caching for speed.
    """
    def __init__(self, X_train, y_train, X_test, y_test, model_name, scorer,
                 feature_names=None, random_state=42):
        self.X_train = np.asarray(X_train)
        self.y_train = np.asarray(y_train)
        self.X_test = np.asarray(X_test)
        self.y_test = np.asarray(y_test)
        self.model_name = model_name
        self.scorer = scorer
        self.data = DataWrapper(list(range(self.X_train.shape[1])), feature_names)
        self.random_state = random_state
        self._cache = {}

    def __call__(self, subset):
        key = tuple(sorted(subset))
        if key in self._cache:
            return self._cache[key]

        if len(key) == 0:
            majority = np.bincount(self.y_train.astype(int)).argmax()
            y_pred = np.full_like(self.y_test, majority)
            score = self.scorer(self.y_test, y_pred)
            self._cache[key] = float(score)
            return float(score)

        model = get_model(self.model_name, self.random_state)
        model.fit(self.X_train[:, key], self.y_train)
        y_pred = model.predict(self.X_test[:, key])
        score = self.scorer(self.y_test, y_pred)
        self._cache[key] = float(score)
        return float(score)


def get_scorer(name):
    if name == "accuracy":
        return lambda y_true, y_pred: accuracy_score(y_true, y_pred)
    elif name == "f1_macro":
        return lambda y_true, y_pred: f1_score(y_true, y_pred, average="macro")
    elif name == "r2":
        return lambda y_true, y_pred: r2_score(y_true, y_pred)
    elif name == "mse":
        return lambda y_true, y_pred: -mean_squared_error(y_true, y_pred)
    elif name == "rmse":
        return lambda y_true, y_pred: -np.sqrt(mean_squared_error(y_true, y_pred))
    elif name == "mae":
        return lambda y_true, y_pred: -mean_absolute_error(y_true, y_pred)
    else:
        raise ValueError(f"Unknown scorer: {name}")


# LOO impacts (retraining paradigm)
def loo_impacts(utility: Utility):
    n = len(utility.data)
    all_set = set(range(n))
    vN = utility(list(all_set))
    impacts = np.zeros(n, dtype=float)
    for i in range(n):
        vN_minus_i = utility(list(all_set - {i}))
        impacts[i] = vN - vN_minus_i
    return float(vN), impacts


# Robustness helpers
def cosine(a, b):
    return float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-12))


def noise_stability_cosine(base_utility: Utility, method_fn, noise_level: float = 0.05):
    phi0 = method_fn(base_utility)

    X_train = base_utility.X_train
    noise = np.random.normal(0, noise_level * X_train.std(axis=0), X_train.shape)
    X_noisy = X_train + noise

    util_noisy = Utility(
        X_noisy, base_utility.y_train, base_utility.X_test, base_utility.y_test,
        model_name=base_utility.model_name,
        scorer=base_utility.scorer,
        feature_names=base_utility.data.data_names,
        random_state=base_utility.random_state + 11
    )
    phi_noisy = method_fn(util_noisy)
    return cosine(phi0, phi_noisy)


def bootstrap_stability_cosine(base_utility: Utility, method_fn, rounds: int = 10):
    X_train = base_utility.X_train
    y_train = base_utility.y_train
    X_test = base_utility.X_test
    y_test = base_utility.y_test

    phi0 = method_fn(base_utility)

    rng = np.random.default_rng(base_utility.random_state)
    n = X_train.shape[0]
    cos_list = []
    for r in range(rounds):
        idx = rng.integers(0, n, size=n)
        Xb = X_train[idx]
        yb = y_train[idx]

        util_b = Utility(
            Xb, yb, X_test, y_test,
            model_name=base_utility.model_name,
            scorer=base_utility.scorer,
            feature_names=base_utility.data.data_names,
            random_state=base_utility.random_state + 1000 + r
        )
        phi_b = method_fn(util_b)
        cos_list.append(cosine(phi0, phi_b))

    return float(np.mean(cos_list))


# Attribution methods
def gately_method_fn(utility: Utility) -> np.ndarray:
    res = exact_gately_value(utility, options=None, progress=False)
    return np.array(res.values, dtype=float)


def least_core_mc_method_fn(utility: Utility) -> np.ndarray:
    res = montecarlo_least_core(
        utility,
        n_iterations=LEAST_CORE_MC_ITERS,
        epsilon=LEAST_CORE_EPSILON,
        options=LEAST_CORE_SOLVER_OPTS,
        progress=False
    )
    return np.array(res.values, dtype=float)


def shapley_mc_method_fn(utility: Utility) -> np.ndarray:
    res = montecarlo_shapley_value(
        utility,
        n_iterations=SHAPLEY_MC_ITERS,
        options=None,
        progress=False
    )
    return np.array(res.values, dtype=float)


ATTRIBUTION_METHODS = {
    "gately": gately_method_fn,
    "least_core_mc": least_core_mc_method_fn,
    "mc_shapley": shapley_mc_method_fn,
}


class TrainRowMaskingEvaluator:
    """
    Masking metrics with TRAIN-row baselines.

    REF_MODE:
      - "pred": reference is model predicted class on original X_test
      - "true": reference is y_test (ground truth)

    Curve signal (DEL/INS/Suff):
      - proba: p(ref)
      - decision_function: signed margin toward ref
      - fallback: 1s (won't crash; avoid relying on this)
    FC signal:
      - proba: logit(p_ref) for binary, log(p_ref) for multiclass
      - decision_function: signed margin
    """

    def __init__(
        self,
        model,
        X_train: np.ndarray,
        X_test: np.ndarray,
        y_test: np.ndarray,
        ref_mode: Literal["pred", "true"] = "pred",
        rng_seed: int = 42,
    ):
        self.model = model
        self.X_train = np.asarray(X_train)
        self.X_test = np.asarray(X_test)
        self.y_test = np.asarray(y_test)
        self.ref_mode = ref_mode
        self.rng = np.random.default_rng(rng_seed)

        self._has_proba = hasattr(model, "predict_proba") and callable(getattr(model, "predict_proba"))
        self._has_decision = hasattr(model, "decision_function") and callable(getattr(model, "decision_function"))
        self._classes = getattr(model, "classes_", None)

        self._ref_idx = self._compute_ref()  # (n_test,) in index space for proba; labels otherwise

        self._orig_curve_signal = self._curve_signal(self.X_test, ref_idx=self._ref_idx)
        self._orig_fc_signal = self._fc_signal(self.X_test, ref_idx=self._ref_idx)

    def _map_labels_to_class_index(self, y: np.ndarray) -> np.ndarray:
        if self._classes is None:
            return y.astype(int)
        cls_to_i = {c: i for i, c in enumerate(self._classes)}
        return np.array([cls_to_i[yy] for yy in y], dtype=int)

    def _compute_ref(self) -> np.ndarray:
        if self._has_proba:
            if self.ref_mode == "pred":
                y_ref = self.model.predict(self.X_test)
            else:
                y_ref = self.y_test
            return self._map_labels_to_class_index(y_ref)
        else:
            # decision-only: keep labels
            if self.ref_mode == "pred":
                return self.model.predict(self.X_test)
            else:
                return self.y_test

    def _curve_signal(self, X: np.ndarray, ref_idx: np.ndarray) -> np.ndarray:
        n = X.shape[0]
        ref = np.asarray(ref_idx)
        if ref.shape[0] != n:
            raise ValueError(f"curve_signal: ref length {ref.shape[0]} != n {n}")

        if self._has_proba:
            proba = self.model.predict_proba(X)
            ref = ref.astype(int)
            return proba[np.arange(n), ref]

        if self._has_decision:
            dec = self.model.decision_function(X)
            if dec.ndim == 1:
                # binary signed margin toward ref (label-based)
                if self._classes is not None and len(self._classes) == 2:
                    pos_label = self._classes[1]
                    sign = np.where(ref == pos_label, 1.0, -1.0)
                else:
                    sign = np.where(ref == 1, 1.0, -1.0)
                return sign * dec
            else:
                # multiclass: need indices; if we don't have, fallback to max
                try:
                    refi = ref.astype(int)
                    return dec[np.arange(n), refi]
                except Exception:
                    return dec.max(axis=1)

        return np.ones(n, dtype=float)

    def _fc_signal(self, X: np.ndarray, ref_idx: np.ndarray) -> np.ndarray:
        eps = 1e-12
        n = X.shape[0]
        ref = np.asarray(ref_idx)
        if ref.shape[0] != n:
            raise ValueError(f"fc_signal: ref length {ref.shape[0]} != n {n}")

        if self._has_proba:
            proba = self.model.predict_proba(X)
            ref = ref.astype(int)
            p_ref = proba[np.arange(n), ref]
            if proba.shape[1] == 2:
                return np.log(p_ref + eps) - np.log(1.0 - p_ref + eps)
            else:
                return np.log(p_ref + eps)

        # decision / fallback: use curve signal (margin)
        return self._curve_signal(X, ref_idx=ref)

    def _sample_train_rows_matrix(self, n_baselines: int, n_test: int) -> np.ndarray:
        idx = self.rng.integers(0, self.X_train.shape[0], size=(n_baselines, n_test))
        return self.X_train[idx]

    def _make_deletion_batch(self, cols_remove: np.ndarray, n_baselines: int) -> np.ndarray:
        n_test, d = self.X_test.shape
        base = self._sample_train_rows_matrix(n_baselines, n_test)
        X_stack = np.repeat(self.X_test[None, :, :], n_baselines, axis=0)
        X_stack[:, :, cols_remove] = base[:, :, cols_remove]
        return X_stack.reshape(n_baselines * n_test, d)

    def _make_insertion_batch(self, cols_keep: np.ndarray, n_baselines: int) -> np.ndarray:
        n_test, d = self.X_test.shape
        base = self._sample_train_rows_matrix(n_baselines, n_test)
        if cols_keep.size > 0:
            base[:, :, cols_keep] = self.X_test[None, :, cols_keep]
        return base.reshape(n_baselines * n_test, d)

    def _batched_mean(self, X_big: np.ndarray, n_baselines: int, which: str, batch_size: int) -> float:
        """
        No tiling assumptions inside chunks.
        We prebuild ref_big for the full (B*n_test) batch, then slice alongside Xc.
        """
        n_test = self.X_test.shape[0]
        ref_big = np.tile(self._ref_idx, n_baselines)  # length B*n_test

        sig_parts = []
        for start in range(0, X_big.shape[0], batch_size):
            end = min(start + batch_size, X_big.shape[0])
            Xc = X_big[start:end]
            refc = ref_big[start:end]

            if which == "curve":
                sig_parts.append(self._curve_signal(Xc, ref_idx=refc))
            elif which == "fc":
                sig_parts.append(self._fc_signal(Xc, ref_idx=refc))
            else:
                raise ValueError("which must be 'curve' or 'fc'")

        sig = np.concatenate(sig_parts, axis=0)
        return float(sig.mean())

    def masking_curve_auc(self, order: np.ndarray, steps: int, mode: str, n_baselines: int, batch_size: int) -> float:
        fracs = np.linspace(0.0, 1.0, steps)
        scores = []
        d = len(order)

        base_curve_score = float(self._orig_curve_signal.mean())

        for f in fracs:
            k = int(round(f * d))
            cols = order[:k] if k > 0 else np.array([], dtype=int)

            if mode == "deletion":
                if cols.size == 0:
                    score = base_curve_score
                else:
                    X_big = self._make_deletion_batch(cols, n_baselines)
                    score = self._batched_mean(X_big, n_baselines=n_baselines, which="curve", batch_size=batch_size)
            elif mode == "insertion":
                X_big = self._make_insertion_batch(cols, n_baselines)
                score = self._batched_mean(X_big, n_baselines=n_baselines, which="curve", batch_size=batch_size)
            else:
                raise ValueError("mode must be 'deletion' or 'insertion'")

            scores.append(score)

        return float(np.trapezoid(np.array(scores), fracs))

    def deletion_auc_drop(self, attribution: np.ndarray, steps: int, n_baselines: int, batch_size: int, rng_seed: int = 42) -> float:
        order_attr = np.argsort(-np.abs(attribution))
        order_rand = np.random.default_rng(rng_seed).permutation(len(attribution))
        auc_attr = self.masking_curve_auc(order_attr, steps, "deletion", n_baselines=n_baselines, batch_size=batch_size)
        auc_rand = self.masking_curve_auc(order_rand, steps, "deletion", n_baselines=n_baselines, batch_size=batch_size)
        return float(auc_rand - auc_attr)

    def insertion_auc_gain(self, attribution: np.ndarray, steps: int, n_baselines: int, batch_size: int, rng_seed: int = 42) -> float:
        order_attr = np.argsort(-np.abs(attribution))
        order_rand = np.random.default_rng(rng_seed).permutation(len(attribution))
        auc_attr = self.masking_curve_auc(order_attr, steps, "insertion", n_baselines=n_baselines, batch_size=batch_size)
        auc_rand = self.masking_curve_auc(order_rand, steps, "insertion", n_baselines=n_baselines, batch_size=batch_size)
        return float(auc_attr - auc_rand)

    def faithfulness_corr(self, attribution: np.ndarray, n_tests: int, subset_size: int, n_baselines: int, batch_size: int) -> float:
        abs_attr = np.abs(attribution)
        d = len(attribution)
        k = min(int(subset_size), max(1, d // 2))
        rng = np.random.default_rng(42)

        base_fc_score = float(self._orig_fc_signal.mean())

        x = np.empty(n_tests, dtype=float)
        y = np.empty(n_tests, dtype=float)

        for t in range(n_tests):
            subset = rng.choice(d, size=k, replace=False)
            x[t] = float(abs_attr[subset].sum())
            X_big = self._make_deletion_batch(subset, n_baselines)
            score_mask = self._batched_mean(X_big, n_baselines=n_baselines, which="fc", batch_size=batch_size)
            y[t] = float(base_fc_score - score_mask)

        corr, _ = pearsonr(x, y)
        return float(corr)

    def sufficiency_multi(self, attribution: np.ndarray, thresholds=(0.8, 0.9, 0.95), n_baselines: int = 8, batch_size: int = 2048) -> Dict[str, float]:
        abs_attr = np.abs(attribution)
        order = np.argsort(-abs_attr)
        d = len(order)

        full_score = float(self._orig_curve_signal.mean())  # positive p_ref (or margin)
        out: Dict[str, float] = {}

        for thr in thresholds:
            target = thr * full_score
            found = False
            for kk in range(1, d + 1):
                cols = order[:kk]
                X_big = self._make_insertion_batch(cols, n_baselines)
                score_k = self._batched_mean(X_big, n_baselines=n_baselines, which="curve", batch_size=batch_size)
                if score_k >= target:
                    out[f"sufficiency_{int(thr*100)}"] = float(kk / d)
                    found = True
                    break
            if not found:
                out[f"sufficiency_{int(thr*100)}"] = 1.0

        return out

    def evaluate_masking_metrics(self, attribution: np.ndarray, steps: int, n_baselines: int, batch_size: int) -> Dict[str, float]:
        return {
            "deletion_auc_drop": self.deletion_auc_drop(attribution, steps, n_baselines=n_baselines, batch_size=batch_size, rng_seed=42),
            "insertion_auc_gain": self.insertion_auc_gain(attribution, steps, n_baselines=n_baselines, batch_size=batch_size, rng_seed=42),
            "faithfulness_corr": self.faithfulness_corr(attribution, n_tests=FC_N_TESTS, subset_size=FC_SUBSET_SIZE, n_baselines=n_baselines, batch_size=batch_size),
            **self.sufficiency_multi(attribution, thresholds=(0.8, 0.9, 0.95), n_baselines=n_baselines, batch_size=batch_size),
        }


# Ranking helpers
def rank_methods(metric_df: pd.DataFrame, higher_is_better_cols):
    rank_df = pd.DataFrame(index=metric_df.index)
    for col in metric_df.columns:
        higher = col in higher_is_better_cols
        rank_df[col] = metric_df[col].rank(ascending=not higher, method="average")
    return rank_df


def rank_summary(rank_df: pd.DataFrame):
    return pd.DataFrame({
        "mean_rank_mu_hat": rank_df.mean(axis=1),
        "std_rank_sigma_hat": rank_df.std(axis=1, ddof=0)
    }).sort_values("mean_rank_mu_hat")


# Main
def main():
    scorer = get_scorer(SCORING)

    master_rows_faith = []
    master_rows_rob = []
    master_rows_faith_rank = []
    master_rows_rob_rank = []
    master_rows_faith_summary = []
    master_rows_rob_summary = []

    for dataset_name in DATASETS:
        X_train, X_test, y_train, y_test, feature_names = load_dataset(dataset_name)

        for model_name in MODEL_NAMES:
            logger.info("\n" + "=" * 70)
            logger.info(f"DATASET: {dataset_name} | MODEL: {model_name}")
            logger.info("=" * 70)

            OUTPUT_DIR = BASE_OUTPUT_DIR / f"{dataset_name}_{model_name}_results"
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

            # retraining-game utility
            utility = Utility(
                X_train, y_train, X_test, y_test,
                model_name=model_name,
                scorer=scorer,
                feature_names=feature_names,
                random_state=RANDOM_STATE
            )

            # Train full model ONCE for masking metrics
            full_model = get_model(model_name, RANDOM_STATE)
            full_model.fit(X_train, y_train)

            evaluator = TrainRowMaskingEvaluator(
                full_model, X_train, X_test, y_test,
                ref_mode=REF_MODE,
                rng_seed=RANDOM_STATE
            )

            vN, impacts = loo_impacts(utility)
            logger.info(f"  v(N) with all features ({SCORING}) = {vN:.4f}")
            logger.info(f"  Masking REF_MODE={REF_MODE} | baselines={PERTURB_N_BASELINES} | batch={PERTURB_BATCH} | FC tests={FC_N_TESTS}")

            # Compute attributions (retraining game)
            attributions = {}
            for mname, mfn in ATTRIBUTION_METHODS.items():
                logger.info(f"Computing attributions: {mname}")
                phi = mfn(utility)
                attributions[mname] = phi

                df_vals = pd.DataFrame({"method": mname, "feature": feature_names, "value": phi})
                df_vals.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_feature_valuations_{mname}.csv", index=False)

                try:
                    plot_single_valuation(
                        phi, feature_names,
                        method_name=mname,
                        title=f"{mname} - {dataset_name}_{model_name} ({SCORING})",
                        output_path=OUTPUT_DIR / f"{dataset_name}_{model_name}_{mname}.png"
                    )
                except Exception as e:
                    logger.warning(f"Plot failed for {mname}: {e}")

            # Faithfulness metrics
            faith_rows = []
            for mname, phi in attributions.items():
                sp, p_sp = spearmanr(phi, impacts)
                pe, p_pe = pearsonr(phi, impacts)

                mask_metrics = evaluator.evaluate_masking_metrics(
                    phi, steps=CURVE_STEPS, n_baselines=PERTURB_N_BASELINES, batch_size=PERTURB_BATCH
                )

                row = {
                    "dataset": dataset_name,
                    "model": model_name,
                    "method": mname,
                    "loo_spearman": float(sp),
                    "loo_pearson": float(pe),
                    "_p_spearman": float(p_sp),
                    "_p_pearson": float(p_pe),
                    "vN": float(vN),
                    **mask_metrics,
                }
                faith_rows.append(row)
                master_rows_faith.append(row)

            faith_df = pd.DataFrame(faith_rows).set_index("method")
            faith_df.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_faithfulness_metrics_raw.csv")
            logger.info("\nFaithfulness metrics (raw):\n" + faith_df.to_string())

            # Robustness metrics
            rob_rows = []
            for mname, mfn in ATTRIBUTION_METHODS.items():
                noise_cos = noise_stability_cosine(utility, mfn, noise_level=NOISE_LEVEL)
                boot_cos = bootstrap_stability_cosine(utility, mfn, rounds=BOOTSTRAP_ROUNDS)
                rob_rows.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "method": mname,
                    "noise_cosine": float(noise_cos),
                    "bootstrap_cosine": float(boot_cos),
                })
                master_rows_rob.append(rob_rows[-1])

            rob_df = pd.DataFrame(rob_rows).set_index("method")
            rob_df.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_robustness_metrics_raw.csv")
            logger.info("\nRobustness metrics (raw):\n" + rob_df.to_string())

            # Ranking
            faith_metric_cols = [
                c for c in faith_df.columns
                if (not c.startswith("_")) and (c not in ["dataset", "model", "vN"])
            ]
            rob_metric_cols = ["noise_cosine", "bootstrap_cosine"]

            faith_rank = pd.DataFrame(index=faith_df.index)
            for col in faith_metric_cols:
                higher = not col.startswith("sufficiency_")  # sufficiency lower is better
                faith_rank[col] = faith_df[col].rank(ascending=not higher, method="average")

            rob_rank = rank_methods(rob_df[rob_metric_cols], higher_is_better_cols=set(rob_metric_cols))

            faith_rank.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_faithfulness_ranks.csv")
            rob_rank.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_robustness_ranks.csv")

            for method in faith_rank.index:
                r = {"dataset": dataset_name, "model": model_name, "method": method}
                r.update({f"rank_{c}": float(faith_rank.loc[method, c]) for c in faith_metric_cols})
                master_rows_faith_rank.append(r)

            for method in rob_rank.index:
                r = {"dataset": dataset_name, "model": model_name, "method": method}
                r.update({f"rank_{c}": float(rob_rank.loc[method, c]) for c in rob_metric_cols})
                master_rows_rob_rank.append(r)

            # Mean rank summary
            faith_summary = rank_summary(faith_rank)
            rob_summary = rank_summary(rob_rank)

            faith_summary.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_faithfulness_rank_summary.csv")
            rob_summary.to_csv(OUTPUT_DIR / f"{dataset_name}_{model_name}_robustness_rank_summary.csv")

            logger.info("\nFaithfulness rank summary (μ̂, σ̂):\n" + faith_summary.to_string())
            logger.info("\nRobustness rank summary (μ̂, σ̂):\n" + rob_summary.to_string())

            for method in faith_summary.index:
                master_rows_faith_summary.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "method": method,
                    "faith_mean_rank_mu_hat": float(faith_summary.loc[method, "mean_rank_mu_hat"]),
                    "faith_std_rank_sigma_hat": float(faith_summary.loc[method, "std_rank_sigma_hat"]),
                })
            for method in rob_summary.index:
                master_rows_rob_summary.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "method": method,
                    "rob_mean_rank_mu_hat": float(rob_summary.loc[method, "mean_rank_mu_hat"]),
                    "rob_std_rank_sigma_hat": float(rob_summary.loc[method, "std_rank_sigma_hat"]),
                })

            logger.info(f"\nSaved per-run outputs to: {OUTPUT_DIR}")
            logger.info("=" * 70)

    # MASTER CSVs
    master_faith = pd.DataFrame(master_rows_faith)
    master_rob = pd.DataFrame(master_rows_rob)
    master_faith_rank = pd.DataFrame(master_rows_faith_rank)
    master_rob_rank = pd.DataFrame(master_rows_rob_rank)
    master_faith_summary = pd.DataFrame(master_rows_faith_summary)
    master_rob_summary = pd.DataFrame(master_rows_rob_summary)

    paper_master = master_faith.merge(master_rob, on=["dataset", "model", "method"], how="outer")
    paper_master = paper_master.merge(master_faith_summary, on=["dataset", "model", "method"], how="outer")
    paper_master = paper_master.merge(master_rob_summary, on=["dataset", "model", "method"], how="outer")

    master_faith_path = BASE_OUTPUT_DIR / "MASTER_faithfulness_raw.csv"
    master_rob_path = BASE_OUTPUT_DIR / "MASTER_robustness_raw.csv"
    master_faith_rank_path = BASE_OUTPUT_DIR / "MASTER_faithfulness_ranks.csv"
    master_rob_rank_path = BASE_OUTPUT_DIR / "MASTER_robustness_ranks.csv"
    paper_master_path = BASE_OUTPUT_DIR / "MASTER_paper_table.csv"

    master_faith.to_csv(master_faith_path, index=False)
    master_rob.to_csv(master_rob_path, index=False)
    master_faith_rank.to_csv(master_faith_rank_path, index=False)
    master_rob_rank.to_csv(master_rob_rank_path, index=False)
    paper_master.to_csv(paper_master_path, index=False)

    logger.info("\nMASTER CSVs saved:")
    logger.info(f"  {master_faith_path}")
    logger.info(f"  {master_rob_path}")
    logger.info(f"  {master_faith_rank_path}")
    logger.info(f"  {master_rob_rank_path}")
    logger.info(f"  {paper_master_path}")


if __name__ == "__main__":
    main()